# 02 — A noise budget cell, end to end

**Runtime:** about 2 minutes on a laptop CPU.

A *budget cell* answers one question: **how much of an observable's fluctuation is this
noise channel responsible for?** This notebook builds two of them end to end — from a
`NoiseConfig` object to a plotted number with error bars — using the **same functions the
shipping budget engine uses**, not a reimplementation.

The two cells:

| channel set | switches on | what it measures |
|---|---|---|
| `all_off` | none (plus `thermal_feedback`) | the deterministic baseline |
| `trn` | `trn` | thermorefractive noise, one-at-a-time (OAT) |

The observable is `u_int_rms_frac` = $\mathrm{std}(U_\mathrm{int})/\mathrm{mean}(U_\mathrm{int})$
over the record — the cheapest of the five, and a pure time-domain statistic with no Welch
segmentation to reason about.

**Common random numbers.** Every channel set is evaluated on the *same* seed list. The
solver's PRNG splits are unconditional, so toggling one channel cannot shift another
channel's stream — which makes the OAT *difference* exact rather than approximate, instead
of being swamped by sampling noise. You will see this directly: `all_off` returns the
identical value for every seed, so its SEM is exactly zero.

## 1. The engine's own pieces

In [1]:
import json, pathlib, sys, tempfile, time, warnings

# Work whether or not the package is installed: notebooks run with cwd=notebooks/.
try:
    import simulator  # noqa: F401
except ModuleNotFoundError:                       # not pip-installed -- fall back to the checkout
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
    import simulator  # noqa: F401
REPO_ROOT = pathlib.Path(simulator.__file__).resolve().parents[1]

import numpy as np
import matplotlib.pyplot as plt

# simulator.lle_solver (imported transitively) forces jax_enable_x64 at import time.
from analysis.dks_access import attach_dispersion, load_cavity_params
from analysis.noise_budget import (
    CHANNEL_SETS,        # name -> NoiseConfig, the 13 sets in table order
    QUICK_RECORDS,       # name -> RecordSpec (t_slow, settle_rt, n_tau, ...)
    RecordContext,       # what an observable needs besides the solver output
    SEED_BASE,           # 100; base of the shared seed list
    budget_sidecar,      # NoiseConfig -> sidecar YAML, verified through the solver
    _metrology_unit,     # settle a DKS under the set, then record it
    obs_u_int_rms_frac,  # the observable
    _summarise,          # mean / SEM / std + bootstrap 95% CI
)

warnings.simplefilter("ignore")

SPEC = QUICK_RECORDS["fast"]
PROBE_MUS = (-100, -10, 0, 10, 100)
SETS = ("all_off", "trn")

cav = attach_dispersion(load_cavity_params(), n_tau=SPEC.n_tau)
print(f"record   : t_slow={SPEC.t_slow} RT, settle={SPEC.settle_rt} RT, n_tau={SPEC.n_tau}")
print(f"duration : {SPEC.t_slow * cav.t_r * 1e6:.4f} us")
print(f"Fourier floor 1/(t_slow*t_r) = {1.0 / (SPEC.t_slow * cav.t_r):.4e} Hz")

[dispersion] measured FSR = 2.445164e+10 Hz vs config 2.460000e+10 Hz (rel diff 0.60%); local D2 = 4.9804e+04 rad/s^2 vs config D2 = 3.7699e+04 rad/s^2.
record   : t_slow=16000 RT, settle=2500 RT, n_tau=256
duration : 0.6504 us
Fourier floor 1/(t_slow*t_r) = 1.5375e+06 Hz


The leading underscore on `_metrology_unit` and `_summarise` is a warning that these are
the engine's internals, not a supported API. They are used here deliberately: the point of
the notebook is that every number below comes from the code that produced the paper's
table. The supported entry point is the command line —

```bash
python analysis/noise_budget.py --quick        # the smoke run, ~13 cells
python analysis/noise_budget.py --seeds 24     # the production run
```

## 2. From `NoiseConfig` to a verified sidecar

`budget_sidecar()` takes the committed device config and makes three edits: it writes the
ECDL flagship amplitude preset into `physical_parameters`, drops the legacy switch keys
(`quantum_noise_enabled`, `pump_noise_enabled`, `fsr_noise_enabled` — which would
*outrank* the `noise:` block), and sets `noise:` to the requested config.

Then it reads the file back through `_resolve_noise_flags`, the solver's own precedence
chain, and hard-errors if any field disagrees. A precedence trap here would not crash —
it would silently produce a complete, plausible budget table for the *wrong* channel sets.

In [2]:
tmpdir = pathlib.Path(tempfile.mkdtemp(prefix="budget_nb_"))
sidecars = {}
for name in SETS:
    nc = CHANNEL_SETS[name]
    sidecars[name] = budget_sidecar(nc, tmpdir / f"{name}.yaml")
    on = nc.enabled_channels or ("none",)
    print(f"{name:8s} sha256={nc.sha256()[:16]}  stochastic channels on: {', '.join(on)}")
    print(f"{'':8s} thermal_feedback={nc.thermal_feedback}  trn_psd_model={nc.trn_psd_model}")

all_off  sha256=ff3b47020db8f8c5  stochastic channels on: none
         thermal_feedback=True  trn_psd_model=kondratiev_gorodetsky
trn      sha256=3073c8e7e0d9c195  stochastic channels on: trn
         thermal_feedback=True  trn_psd_model=kondratiev_gorodetsky


`thermal_feedback` is pinned **on** in every set. It is the deterministic thermo-optic ODE
rather than a noise channel, so letting it differ between `all_off` and `all_on` would
contaminate every OAT contribution with the thermal dynamics.

## 3. Run the cells

`_metrology_unit` settles a single DKS **under the same channel set** as the record —
settling noise-free and only then switching channels on would put a transient into the
first decade of every spectrum — then records `t_slow` round trips from that settled state
using a distinct trajectory key.

In [3]:
seeds2 = [SEED_BASE + k for k in range(2)]      # the committed --quick seed list

def run_cell(name, seeds):
    # NoiseConfig -> per-seed observable values, via the shipping engine.
    values = []
    for seed in seeds:
        solution = _metrology_unit(SPEC, cav, sidecars[name], seed, probe_mus=PROBE_MUS)
        ctx = RecordContext(
            record="fast", t_r=cav.t_r, kappa=cav.kappa, t_slow=SPEC.t_slow,
            snapshot_interval=SPEC.snapshot_interval, n_tau=SPEC.n_tau,
            probe_mus=PROBE_MUS, channel_set=name, seed=seed,
        )
        point, unit, err = obs_u_int_rms_frac(solution, ctx)
        values.append(point["value"])
    return values

t0 = time.time()
per_seed2 = {name: run_cell(name, seeds2) for name in SETS}
print(f"2 seeds x {len(SETS)} sets in {time.time() - t0:.0f} s\n")
for name in SETS:
    print(f"{name:8s} per-seed: {[f'{v:.6e}' for v in per_seed2[name]]}")

2 seeds x 2 sets in 14 s

all_off  per-seed: ['1.395032e-04', '1.395032e-04']
trn      per-seed: ['1.666504e-04', '1.864220e-04']


Look at `all_off`: **both seeds give byte-identical values.** With every stochastic channel
off there is nothing for the seed to vary, so the deterministic baseline has exactly zero
spread by construction. That is the common-random-numbers guarantee showing itself.

## 4. Check against the committed table

`analysis/results/budget/budget.json` is the committed `--quick` run. We just recomputed
two of its cells from scratch; they should match.

In [4]:
committed = json.loads(
    (REPO_ROOT / "analysis" / "results" / "budget" / "budget.json").read_text(encoding="utf-8")
)
prov = committed["provenance"]
print(f"committed run: seeds={committed['run']['seed_list']}  quick={committed['run']['quick']}")
print(f"generated {prov.get('generated_utc', '?')} at commit {prov.get('git_commit', '?')}\n")

for name in SETS:
    ref = committed["cells"][name]["u_int_rms_frac"]["points"]["value"]["statistics"]
    mine = _summarise(per_seed2[name], seeds2)
    rel = abs(mine["mean"] - ref["mean"]) / ref["mean"]
    print(f"{name:8s} recomputed={mine['mean']:.9e}  committed={ref['mean']:.9e}  "
          f"rel.diff={rel:.2e}")
    # Loose gate: exact on the reference toolchain, but XLA may reassociate
    # reductions on different hardware. See docs/LIMITATIONS.md.
    assert rel < 1e-6, f"{name}: recomputed cell disagrees with the committed table"
print("\nboth cells reproduce the committed table")

committed run: seeds=[100, 101]  quick=True
generated 2026-08-17T07:38:45Z at commit 4880653e7966

all_off  recomputed=1.395031679e-04  committed=1.395031679e-04  rel.diff=0.00e+00
trn      recomputed=1.765362281e-04  committed=1.765362281e-04  rel.diff=0.00e+00

both cells reproduce the committed table


## 5. More seeds, and the bootstrap CI

Two seeds is a smoke test, not a measurement. `_summarise` reports the mean, the SEM and a
percentile bootstrap 95 % CI over 2000 resamples; with two points the CI is meaningless, so
extend the shared seed list to six — **the same six for both sets**, which is what keeps the
OAT difference exact.

In [5]:
N_SEEDS = 6
seeds6 = [SEED_BASE + k for k in range(N_SEEDS)]

t0 = time.time()
per_seed6 = {name: run_cell(name, seeds6) for name in SETS}
stats = {name: _summarise(per_seed6[name], seeds6) for name in SETS}
print(f"{N_SEEDS} seeds x {len(SETS)} sets in {time.time() - t0:.0f} s\n")

for name in SETS:
    s = stats[name]
    lo, hi = s["ci95_mean"]
    print(f"{name:8s} mean={s['mean']:.6e}  SEM={s['sem']:.3e}  "
          f"95% CI=[{lo:.6e}, {hi:.6e}]  (n={s['n_seeds_used']}, "
          f"{s['bootstrap_resamples']} resamples)")

6 seeds x 2 sets in 32 s

all_off  mean=1.395032e-04  SEM=0.000e+00  95% CI=[1.395032e-04, 1.395032e-04]  (n=6, 2000 resamples)
trn      mean=1.742991e-04  SEM=4.946e-06  95% CI=[1.656749e-04, 1.828301e-04]  (n=6, 2000 resamples)


In [6]:
base = stats["all_off"]["mean"]
trn = stats["trn"]["mean"]
oat = trn - base
oat_err = stats["trn"]["sem"]          # all_off has exactly zero spread, so it adds none

print(f"deterministic baseline      : {base:.6e}")
print(f"with thermorefractive noise : {trn:.6e}")
print(f"OAT contribution of `trn`   : {oat:+.6e} +/- {oat_err:.3e}"
      f"   ({100 * oat / base:+.1f}% of baseline)")
print()
print(f"all_off SEM = {stats['all_off']['sem']:.3e}  <- exactly zero: deterministic")

deterministic baseline      : 1.395032e-04
with thermorefractive noise : 1.742991e-04
OAT contribution of `trn`   : +3.479589e-05 +/- 4.946e-06   (+24.9% of baseline)

all_off SEM = 0.000e+00  <- exactly zero: deterministic


## 6. The plot

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.0))

names = list(SETS)
means = [stats[n]["mean"] for n in names]
errs = np.array([[stats[n]["mean"] - stats[n]["ci95_mean"][0] for n in names],
                 [stats[n]["ci95_mean"][1] - stats[n]["mean"] for n in names]])
ax[0].bar(names, means, yerr=errs, capsize=6, color=["0.6", "C0"], edgecolor="k", lw=0.8)
ax[0].set(ylabel=r"$\mathrm{std}(U_{\rm int})/\mathrm{mean}(U_{\rm int})$",
          title=f"Budget cells (mean, bootstrap 95% CI, n={N_SEEDS})")
ax[0].grid(alpha=0.3, axis="y")

for i, name in enumerate(names):
    ax[1].plot(seeds6, per_seed6[name], "o-", ms=5, lw=1.0,
               color=["0.4", "C0"][i], label=name)
ax[1].set(xlabel="seed", ylabel="per-seed value",
          title="Common random numbers: same seeds, both sets")
ax[1].legend()
ax[1].grid(alpha=0.3)
fig.tight_layout()

The flat grey line in the right panel is the whole point: with every channel off, the
observable does not depend on the seed at all. Its spread is not "small" — it is zero, so
the OAT difference carries no baseline sampling error.

## 7. What this is, and what it is not

* **This is a `--quick` record.** `t_slow = 16 000` round trips is 0.65 µs, with a Fourier
  floor near 1.5 MHz. It is a plumbing measurement. The production records are
  `t_slow = 200 000` (fast) and `2 x 10^7` (slow); a target below a record's floor is
  reported as `null` with the record length that *would* observe it, never interpolated to
  the nearest observed bin.
* **Cells from different records are not comparable.** Every number in the budget table
  carries the record that produced it, for exactly this reason. See
  `docs/LIMITATIONS.md` on record-length dependence.
* **`pyro_eo` and `fsr` are never run alone.** They are driven by the *same* $\delta T(t)$
  realization as `trn`, so `pyro_eo=True, trn=False` is identically zero and the solver
  warns about it. Their rows carry `trn`; the physically meaningful grouped channel is
  `dT_family`.
* **Reproduce the full table** with:

```bash
python analysis/noise_budget.py --quick        # what this notebook recomputed, ~13 cells
python analysis/noise_budget.py --seeds 24     # the production budget
```

Both write `analysis/results/budget/budget.json`, `budget_table.md` and `budget_table.tex`,
checkpoint after every solver unit, and support `--resume`.